# 05 — Shaft Sizing, Bearings & Campbell Diagram

**Purpose:** Complete the mechanical design of the rotating assembly. Size the shaft from both strength and stiffness criteria, select SKF catalogue bearings to meet L10 life targets, and build the Campbell diagram to confirm all engine-order crossings are managed.

**Inputs:** IGV geometry and rotor meanline from notebooks 03 and 04.

**Outputs:**
- Rotor mass and polar moment of inertia (thin-rim disc model)
- Shaft diameter governed by critical speed margin ≥ 30%
- Bearing A (locating, angular contact) and Bearing B (free, DGBB) selections
- Campbell diagram with all engine-order crossings flagged
- VFD skip-band recommendations

**Layout:** fixed–free, between-bearings: `Motor ── Coupling ── [Bearing A] ── Rotor ── [Bearing B]`

**References:** Shigley & Mischke (2014) Ch. 6–7; SKF General Catalogue (2018); ISO 21940-11:2016; ISO 281:2007; Rankine (1869).

In [ ]:
import sys, pathlib

# Locate repo root (directory that contains src/) regardless of
# where the notebook file sits (repo root or notebooks/ subfolder)
_here = pathlib.Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "src").is_dir()),
    _here,
)
sys.path.insert(0, str(_root))
pathlib.Path(_root / "figures").mkdir(exist_ok=True)


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.igv   import igv_geometry, meanline_with_igv
from src.shaft import (rotor_mass, shaft_sizing, bearing_selection,
                       campbell_data, print_shaft_summary,
                       print_bearing_summary, print_campbell_summary)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Load design point from notebook 01 ─────────────────────
import json as _json, pathlib as _pl

_dp_path = _pl.Path(_root / "design_point.json")
if not _dp_path.exists():
    raise FileNotFoundError(
        "design_point.json not found — run notebook 01 first.")

dp = _json.loads(_dp_path.read_text())

# Unpack for convenience
D_TIP = dp["D_tip"]
N_RPM = dp["N_RPM"]
PR    = dp["PR"]
NU    = dp["nu"]
PHI   = dp["phi"]
ETA   = dp["eta_is"]
print(f"Design point: D_tip={D_TIP*1000:.0f} mm  N={N_RPM} RPM  PR={PR}  nu={NU}  phi={PHI}  eta={ETA}")


## 1. Inputs from aerodynamic design

In [ ]:
igv   = igv_geometry(D_tip=D_TIP, nu=NU, N_RPM=N_RPM, phi=PHI, alpha1_deg=0.0)
rotor = meanline_with_igv(igv, PR=PR, eta_is=ETA)

print(f'P_shaft   = {rotor["P_shaft_kW"]:.1f} kW')
print(f'Rotor ΔP₀ = {rotor["dP0_rotor_Pa"]:.0f} Pa')
print(f'Annulus A = {igv["A_annulus_m2"]*1e4:.2f} cm²')

## 2. Rotor mass — thin-rim disc model (Al 7075-T6)

In [ ]:
rr = rotor_mass(
    r_tip     = igv['r_tip_mm'] / 1000,
    r_hub     = igv['r_hub_mm'] / 1000,
    h_blade   = igv['h_annulus_mm'] / 1000,
    B         = igv['B_blades'],
    chord_mid = igv['rotor_chord_mid_mm'] / 1000,
)

print('Rotor mass breakdown:')
print(f'  Disc rim      : {rr["m_rim_kg"]:.2f} kg')
print(f'  Disc web      : {rr["m_web_kg"]:.2f} kg')
print(f'  Blades (B={rr["B"]}) : {rr["m_blades_kg"]:.2f} kg')
print(f'  ─────────────────')
print(f'  Total         : {rr["m_total_kg"]:.2f} kg')
print(f'  J_zz          : {rr["J_zz_kgm2"]:.3f} kg·m²')

## 3. Shaft sizing — strength vs stiffness criterion

In [ ]:
sr = shaft_sizing(
    rr,
    igv,
    rotor,
    L_span   = 0.700,   # [m] bearing span
    G_grade  = 1.0,     # ISO 21940 balance grade G1.0 (precision lab rig)
)

print_shaft_summary(sr, rr)

## 4. Shaft diameter sensitivity to bearing span

In [ ]:
span_range = np.linspace(0.500, 1.000, 30)
d_str_s, d_sti_s, d_s, Ncr_s = [], [], [], []

for L in span_range:
    s = shaft_sizing(rr, igv, rotor, L_span=L)
    d_str_s.append(s['d_strength_mm'])
    d_sti_s.append(s['d_adopt_mm'])
    Ncr_s.append(s['N_cr_RPM'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Shaft Sizing vs Bearing Span', fontweight='bold')

axes[0].plot(span_range*1000, d_str_s, '#D85A30', lw=2, label='Strength requirement')
axes[0].plot(span_range*1000, d_sti_s, '#185FA5', lw=2, label='Stiffness requirement (adopted)')
axes[0].axvline(700, color='gray', ls=':', label='L = 700 mm chosen')
axes[0].set(xlabel='Bearing span [mm]', ylabel='Shaft diameter [mm]')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.35)

axes[1].plot(span_range*1000, Ncr_s, '#1D9E75', lw=2)
axes[1].axhline(3500, color='#D85A30', ls='--', lw=1.5, label='N_op = 3500 RPM')
axes[1].axhline(3500*1.30, color='red', ls=':', lw=1.5, label='N_op × 1.30 target')
axes[1].axvline(700, color='gray', ls=':')
axes[1].set(xlabel='Bearing span [mm]', ylabel='First critical speed [RPM]')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/05_shaft_sensitivity.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Bearing selection (SKF catalogue, L10 = 20 000 h)

In [ ]:
br = bearing_selection(sr, L10_target_h=20000)
print_bearing_summary(br)

## 6. Campbell diagram

In [ ]:
cd = campbell_data(sr, igv)
print_campbell_summary(cd)

In [ ]:
N_arr  = cd['N_range']
f_n1   = cd['f_n1_Hz']
N_op   = cd['N_op_RPM']
N_cr   = cd['N_cr_RPM']
EO_list = cd['EO_list']
B = cd['B']; V = cd['V']

fig, ax = plt.subplots(figsize=(10, 6))

colors_eo = plt.cm.tab10(np.linspace(0, 1, len(EO_list)))

for eo, col in zip(EO_list, colors_eo):
    f_eo = eo * N_arr / 60
    lw   = 2.2 if eo in (1, B, V) else 1.0
    ls   = '-' if eo in (1, B, V) else '--'
    lbl  = f'{eo}× EO' + (f' (B={B})' if eo==B else '') + (f' (V={V})' if eo==V else '')
    ax.plot(N_arr, f_eo, color=col, lw=lw, ls=ls, label=lbl)

# Natural frequency (horizontal line — Rankine approximation)
ax.axhline(f_n1, color='black', lw=2.5, label=f'1st bending mode {f_n1:.1f} Hz')

# Operating speed
ax.axvline(N_op, color='#185FA5', lw=2, ls='-.',
           label=f'N_op = {N_op} RPM')

# Mark crossings
for c in cd['crossings']:
    col_x = 'red' if c['below_op'] and c['EO'] <= 6 else 'orange'
    ax.plot(c['N_cross'], c['f_cross_Hz'], 'o', ms=8, color=col_x, zorder=5)
    ax.annotate(f" {c['EO']}×\n {c['N_cross']:.0f}",
                xy=(c['N_cross'], c['f_cross_Hz']),
                fontsize=7.5, color=col_x)

ax.set(xlabel='Rotational speed [RPM]', ylabel='Frequency [Hz]',
       title='Campbell Diagram — First Bending Mode',
       xlim=(0, N_op * 1.25),
       ylim=(0, f_n1 * 1.6))
ax.legend(fontsize=8, ncol=2, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/05_campbell_diagram.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. VFD skip-band recommendations

Engine-order crossings below operating speed must be traversed on every start/stop. Programme skip bands in the VFD to minimise dwell time at these speeds.

In [ ]:
skip_band_rpm = 200  # ±200 RPM per crossing

print('VFD skip-band schedule')
print('─' * 50)
print(f'  {"EO":>4}  {"N_crossing [RPM]":>18}  {"Skip band":>18}')
print('  ' + '─' * 44)

below = [c for c in cd['crossings'] if c['below_op']]
for c in sorted(below, key=lambda x: x['N_cross']):
    lo = c['N_cross'] - skip_band_rpm
    hi = c['N_cross'] + skip_band_rpm
    print(f'  {c["EO"]:>4}×  {c["N_cross"]:>16.0f}  {lo:.0f} – {hi:.0f} RPM  {c["note"]}')

if not below:
    print('  No crossings below operating speed ✓')

---
**Proceed to** `06_blade_design.ipynb`.